In [3]:
from scipy.io import loadmat

mat = loadmat(r".\data\reference\original_mat\eeg_record1.mat")

print(mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'o'])


In [7]:
from scipy.io import loadmat

mat = loadmat(r".\data\reference\original_mat\eeg_record1.mat")
o = mat["o"]

print("o.shape:", o.shape)
print("o.dtype:", o.dtype)
print("o.dtype.names:", o.dtype.names)

o.shape: (1, 1)
o.dtype: [('id', 'O'), ('tag', 'O'), ('nS', 'O'), ('sampFreq', 'O'), ('marker', 'O'), ('timestamp', 'O'), ('data', 'O'), ('trials', 'O')]
o.dtype.names: ('id', 'tag', 'nS', 'sampFreq', 'marker', 'timestamp', 'data', 'trials')


In [9]:
from scipy.io import loadmat

mat = loadmat(r".\data\reference\original_mat\eeg_record1.mat")
o = mat["o"]

for name in o.dtype.names:
    field = o[name][0, 0]
    print("=" * 50)
    print("字段名:", name)
    print("类型:", type(field))
    
    if hasattr(field, "shape"):
        print("形状:", field.shape)
        print("数据类型:", field.dtype)
        print("前几个值:", field.flatten()[:10])
    else:
        print("值:", field)
        

字段名: id
类型: <class 'numpy.ndarray'>
形状: (1,)
数据类型: <U21
前几个值: ['201410092013.D091BB44']
字段名: tag
类型: <class 'numpy.ndarray'>
形状: (0,)
数据类型: <U1
前几个值: []
字段名: nS
类型: <class 'numpy.ndarray'>
形状: (1, 1)
数据类型: int32
前几个值: [308868]
字段名: sampFreq
类型: <class 'numpy.ndarray'>
形状: (1, 1)
数据类型: uint8
前几个值: [128]
字段名: marker
类型: <class 'numpy.ndarray'>
形状: (308868, 1)
数据类型: uint8
前几个值: [0 0 0 0 0 0 0 0 0 0]
字段名: timestamp
类型: <class 'numpy.ndarray'>
形状: (308868, 6)
数据类型: float64
前几个值: [2.014e+03 1.000e+01 9.000e+00 2.000e+01 1.400e+01 1.536e+00 2.014e+03
 1.000e+01 9.000e+00 2.000e+01]
字段名: data
类型: <class 'numpy.ndarray'>
形状: (308868, 25)
数据类型: float64
前几个值: [3.00000000e+00 0.00000000e+00 4.63000000e+02 4.44000000e+03
 4.41794872e+03 5.39076923e+03 3.83384615e+03 4.01948718e+03
 4.65641026e+03 4.74205128e+03]
字段名: trials
类型: <class 'numpy.ndarray'>
形状: (1, 2, 14, 128)
数据类型: float64
前几个值: [4441.02564103 4441.02564103 4440.         4441.02564103 4440.51282051
 4440.51282051 4443.07692308 4442.5641

In [11]:
from scipy.io import loadmat
import pandas as pd
import numpy as np

# =========================
# 1. 配置区
# =========================

file_path = r".\data\reference\original_mat\eeg_record1.mat"   # 如果你文件名没有 .mat，就改成 "eeg_record1"
output_excel = r".\artifacts\legacy\notebook_outputs\reference_inspection\mat_structure_preview.xlsx"

# 每个大数据表最多导出多少行，防止 Excel 太卡
MAX_PREVIEW_ROWS = 5000

# data 预览最多导出多少列
MAX_PREVIEW_COLS = 50


# =========================
# 2. 工具函数
# =========================

def safe_shape(x):
    """安全获取 shape"""
    return getattr(x, "shape", None)


def safe_dtype(x):
    """安全获取 dtype"""
    return getattr(x, "dtype", None)


def get_mat_field(struct_obj, field_name):
    """
    适配 MATLAB 结构体字段读取。
    你的 o 是 1x1 struct，所以字段一般要用 o[field][0,0] 取。
    """
    return struct_obj[field_name][0, 0]


def preview_array(arr, max_items=10):
    """预览前几个值"""
    try:
        return arr.flatten()[:max_items]
    except Exception:
        return str(arr)[:200]


def make_2d_dataframe(arr, max_rows=5000, max_cols=50, prefix="col"):
    """
    把数组尽量变成二维 DataFrame。
    只预览部分，避免 Excel 爆炸。
    """
    arr = np.asarray(arr)

    if arr.ndim == 0:
        return pd.DataFrame([[arr.item()]])

    elif arr.ndim == 1:
        preview = arr[:max_rows]
        return pd.DataFrame(preview, columns=[prefix])

    elif arr.ndim == 2:
        preview = arr[:max_rows, :max_cols]
        columns = [f"{prefix}_{i}" for i in range(preview.shape[1])]
        return pd.DataFrame(preview, columns=columns)

    else:
        # 高维数据不直接全部展开，只返回形状信息
        return pd.DataFrame({
            "说明": [f"这是 {arr.ndim} 维数组，不能直接完整当 Excel 二维表看"],
            "shape": [str(arr.shape)],
            "dtype": [str(arr.dtype)]
        })


def summarize_field(name, value):
    """生成字段摘要"""
    shape = safe_shape(value)
    dtype = safe_dtype(value)

    if isinstance(value, np.ndarray):
        preview = preview_array(value)
        preview_text = str(preview)
    else:
        preview_text = str(value)[:200]

    return {
        "字段名": name,
        "Python类型": str(type(value)),
        "shape": str(shape),
        "dtype": str(dtype),
        "前几个值": preview_text
    }


# =========================
# 3. 读取 mat 文件
# =========================

mat = loadmat(file_path)

print("mat 文件顶层变量：")
print(mat.keys())

# 去掉 MATLAB 自带字段
valid_keys = [k for k in mat.keys() if not k.startswith("__")]

print("\n有效变量：")
print(valid_keys)


# =========================
# 4. 生成顶层结构摘要
# =========================

top_summary = []

for key in valid_keys:
    value = mat[key]
    top_summary.append({
        "变量名": key,
        "Python类型": str(type(value)),
        "shape": str(safe_shape(value)),
        "dtype": str(safe_dtype(value)),
        "是否结构体字段": str(value.dtype.names if hasattr(value, "dtype") else None)
    })

top_summary_df = pd.DataFrame(top_summary)


# =========================
# 5. 针对你的文件：读取 o 结构体
# =========================

if "o" not in mat:
    raise KeyError("这个 mat 文件里没有变量 o，请先看 valid_keys 里真正的变量名。")

o = mat["o"]

print("\n变量 o 的信息：")
print("o.shape:", o.shape)
print("o.dtype:", o.dtype)
print("o.dtype.names:", o.dtype.names)

field_names = o.dtype.names

field_summary = []

for name in field_names:
    field = get_mat_field(o, name)

    print("=" * 60)
    print("字段名:", name)
    print("类型:", type(field))
    print("形状:", safe_shape(field))
    print("数据类型:", safe_dtype(field))
    print("前几个值:", preview_array(field))

    field_summary.append(summarize_field(name, field))

field_summary_df = pd.DataFrame(field_summary)


# =========================
# 6. 提取关键字段
# =========================

data = get_mat_field(o, "data")
marker = get_mat_field(o, "marker")
timestamp = get_mat_field(o, "timestamp")
trials = get_mat_field(o, "trials")
sampFreq = get_mat_field(o, "sampFreq")
nS = get_mat_field(o, "nS")

# 把采样率、采样点数变成普通数字
fs = int(np.asarray(sampFreq).squeeze())
n_samples = int(np.asarray(nS).squeeze())

print("\n关键字段：")
print("采样率 fs:", fs)
print("采样点数 nS:", n_samples)
print("data.shape:", data.shape)
print("marker.shape:", marker.shape)
print("timestamp.shape:", timestamp.shape)
print("trials.shape:", trials.shape)


# =========================
# 7. 制作 data 预览表
# =========================

# data 是 308868 × 25
# 一般理解为：行 = 时间点，列 = 通道/特征
data_preview = data[:MAX_PREVIEW_ROWS, :MAX_PREVIEW_COLS]

data_df = pd.DataFrame(
    data_preview,
    columns=[f"data_col_{i}" for i in range(data_preview.shape[1])]
)

# 添加时间秒
time_s = np.arange(data_preview.shape[0]) / fs
data_df.insert(0, "time_s", time_s)

# 添加 marker
marker_flat = marker.reshape(-1)
data_df.insert(1, "marker", marker_flat[:data_preview.shape[0]])


# =========================
# 8. 制作 timestamp 预览表
# =========================

timestamp_preview = timestamp[:MAX_PREVIEW_ROWS, :]

timestamp_df = pd.DataFrame(
    timestamp_preview,
    columns=["year", "month", "day", "hour", "minute", "second"]
)


# =========================
# 9. 制作 marker 预览表
# =========================

marker_df = pd.DataFrame({
    "sample_index": np.arange(min(MAX_PREVIEW_ROWS, len(marker_flat))),
    "time_s": np.arange(min(MAX_PREVIEW_ROWS, len(marker_flat))) / fs,
    "marker": marker_flat[:MAX_PREVIEW_ROWS]
})


# =========================
# 10. 制作 trials 预览
# =========================

trials_info_df = pd.DataFrame({
    "说明": [
        "trials 是高维数组，不能直接完整当普通二维 Excel 表看",
        "你的 trials.shape 是",
        "最后一维 128 很可能表示 128 个采样点",
        "因为采样率 fs=128，所以 128 个点大约是 1 秒",
    ],
    "值": [
        "",
        str(trials.shape),
        "",
        "",
    ]
})

# 你的 trials.shape 是 (1, 2, 14, 128)
# 取第 0 组、第 0 类，得到 14 × 128
trial_0_0 = trials[0, 0, :, :]

trial_0_0_df = pd.DataFrame(
    trial_0_0,
    index=[f"ch_{i}" for i in range(trial_0_0.shape[0])],
    columns=[f"t_{i}" for i in range(trial_0_0.shape[1])]
)

# 再取第 0 组、第 1 类
trial_0_1 = trials[0, 1, :, :]

trial_0_1_df = pd.DataFrame(
    trial_0_1,
    index=[f"ch_{i}" for i in range(trial_0_1.shape[0])],
    columns=[f"t_{i}" for i in range(trial_0_1.shape[1])]
)


# =========================
# 11. 导出 Excel
# =========================

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    top_summary_df.to_excel(writer, sheet_name="01_mat顶层结构", index=False)
    field_summary_df.to_excel(writer, sheet_name="02_o字段结构", index=False)
    data_df.to_excel(writer, sheet_name="03_data预览", index=False)
    timestamp_df.to_excel(writer, sheet_name="04_timestamp预览", index=False)
    marker_df.to_excel(writer, sheet_name="05_marker预览", index=False)
    trials_info_df.to_excel(writer, sheet_name="06_trials说明", index=False)
    trial_0_0_df.to_excel(writer, sheet_name="07_trial_0_0预览")
    trial_0_1_df.to_excel(writer, sheet_name="08_trial_0_1预览")

print("\n完成！")
print(f"已经生成 Excel 文件：{output_excel}")
print("你可以在 VS Code 左侧文件列表里找到它，然后右键 Reveal in File Explorer 打开。")

mat 文件顶层变量：
dict_keys(['__header__', '__version__', '__globals__', 'o'])

有效变量：
['o']

变量 o 的信息：
o.shape: (1, 1)
o.dtype: [('id', 'O'), ('tag', 'O'), ('nS', 'O'), ('sampFreq', 'O'), ('marker', 'O'), ('timestamp', 'O'), ('data', 'O'), ('trials', 'O')]
o.dtype.names: ('id', 'tag', 'nS', 'sampFreq', 'marker', 'timestamp', 'data', 'trials')
字段名: id
类型: <class 'numpy.ndarray'>
形状: (1,)
数据类型: <U21
前几个值: ['201410092013.D091BB44']
字段名: tag
类型: <class 'numpy.ndarray'>
形状: (0,)
数据类型: <U1
前几个值: []
字段名: nS
类型: <class 'numpy.ndarray'>
形状: (1, 1)
数据类型: int32
前几个值: [308868]
字段名: sampFreq
类型: <class 'numpy.ndarray'>
形状: (1, 1)
数据类型: uint8
前几个值: [128]
字段名: marker
类型: <class 'numpy.ndarray'>
形状: (308868, 1)
数据类型: uint8
前几个值: [0 0 0 0 0 0 0 0 0 0]
字段名: timestamp
类型: <class 'numpy.ndarray'>
形状: (308868, 6)
数据类型: float64
前几个值: [2.014e+03 1.000e+01 9.000e+00 2.000e+01 1.400e+01 1.536e+00 2.014e+03
 1.000e+01 9.000e+00 2.000e+01]
字段名: data
类型: <class 'numpy.ndarray'>
形状: (308868, 25)
数据类型: float64
前几个值: [3.0

PermissionError: [Errno 13] Permission denied: 'mat_structure_preview.xlsx'

In [12]:
from scipy.io import loadmat
import pandas as pd
import numpy as np

# =========================
# 1. 读取 mat 文件
# =========================

mat = loadmat(r".\data\reference\original_mat\eeg_record1.mat")   # 如果你文件没有 .mat 后缀，就改成 "eeg_record1"

# 取出真正要用的 data
eeg_data = mat["o"]["data"][0][0]

print("eeg_data shape:", eeg_data.shape)
print("eeg_data dtype:", eeg_data.dtype)

# =========================
# 2. 转成 Excel 表格
# =========================

# 生成列名：data_col_0 ~ data_col_24
columns = [f"data_col_{i}" for i in range(eeg_data.shape[1])]

df = pd.DataFrame(eeg_data, columns=columns)

# 加一列 sample_index，表示第几个采样点
df.insert(0, "sample_index", np.arange(eeg_data.shape[0]))

# 如果你想加时间秒数，需要采样率
fs = int(mat["o"]["sampFreq"][0][0].squeeze())   # 你的 fs 是 128
df.insert(1, "time_s", df["sample_index"] / fs)

# =========================
# 3. 先在 Jupyter 里预览
# =========================

display(df.head(20))

# =========================
# 4. 导出 Excel
# =========================

# 推荐先导出前 5000 行，打开不会太卡
df.head(5000).to_excel(r".\artifacts\legacy\notebook_outputs\reference_inspection\eeg_data_preview_5000.xlsx", index=False)

print("已生成：eeg_data_preview_5000.xlsx")

eeg_data shape: (308868, 25)
eeg_data dtype: float64


,sample_index,time_s,data_col_0,data_col_1,data_col_2,data_col_3,data_col_4,data_col_5,data_col_6,data_col_7,...,data_col_15,data_col_16,data_col_17,data_col_18,data_col_19,data_col_20,data_col_21,data_col_22,data_col_23,data_col_24
0,0,0.000000,3.0,0.0,463.0,4440.000000,4417.948718,5390.769231,3833.846154,4019.487179,...,4335.384615,4563.589744,1573.0,1726.0,764.304,0.031277,0.0,0.0,0.0,0.0
1,1,0.007812,4.0,0.0,0.0,4439.487179,4417.948718,5389.230769,3830.256410,4020.000000,...,4331.794872,4566.666667,1570.0,1727.0,764.304,0.031277,0.0,0.0,0.0,0.0
2,2,0.015625,5.0,0.0,447.0,4438.974359,4414.871795,5385.641026,3829.743590,4017.948718,...,4333.846154,4557.948718,1567.0,1726.0,764.304,0.031277,0.0,0.0,0.0,0.0
3,3,0.023438,6.0,0.0,500.0,4438.974359,4410.256410,5381.025641,3831.794872,4016.410256,...,4334.358974,4552.820513,1564.0,1723.0,764.304,0.031277,0.0,0.0,0.0,0.0
4,4,0.031250,7.0,0.0,424.0,4439.487179,4407.179487,5378.461538,3831.282051,4019.487179,...,4334.358974,4546.666667,1565.0,1720.0,764.304,0.031277,0.0,0.0,0.0,0.0
5,5,0.039062,8.0,0.0,418.0,4440.512821,4398.974359,5372.307692,3831.282051,4021.025641,...,4335.384615,4537.435897,1564.0,1720.0,764.304,0.031277,0.0,0.0,0.0,0.0
6,6,0.046875,9.0,0.0,0.0,4438.974359,4394.871795,5366.153846,3831.282051,4019.487179,...,4333.846154,4533.333333,1568.0,1719.0,764.304,0.031277,0.0,0.0,0.0,0.0
7,7,0.054688,10.0,0.0,0.0,4437.948718,4396.410256,5362.564103,3828.717949,4017.948718,...,4334.358974,4531.282051,1569.0,1720.0,764.304,0.031277,0.0,0.0,0.0,0.0
8,8,0.062500,11.0,0.0,449.0,4438.974359,4390.769231,5360.512821,3827.179487,4017.948718,...,4335.897436,4532.307692,1572.0,1719.0,764.304,0.031277,0.0,0.0,0.0,0.0
9,9,0.070312,12.0,0.0,0.0,4437.948718,4382.564103,5362.051282,3828.205128,4018.974359,...,4332.307692,4540.512821,1572.0,1720.0,764.304,0.031277,0.0,0.0,0.0,0.0


已生成：eeg_data_preview_5000.xlsx
